# TD1 — Training and evaluating a classical model

**Machine Learning and AI for Analytical Chemistry — Part 1 (Foundations)**

In this first hands-on session you will train and evaluate classical machine
learning models on a real analytical-chemistry dataset: the chemical analysis
of wines from three cultivars.

By the end you should be able to:
- load a dataset and inspect it as a feature table,
- split data into training and test sets,
- train a decision tree, a random forest, and an SVM,
- read accuracy, a confusion matrix, and feature importances,
- see why feature scaling matters, and
- run k-means as an unsupervised exploration.

> Proper validation (cross-validation, leakage, uncertainty) is the topic of
> Part 3. Here we keep evaluation deliberately simple: a single train/test
> split, just enough to compare models.

This notebook runs as-is on Google Colab. Run the cells from top to bottom.


## 0. Setup

All packages below ship with Colab. Just run this cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete.")

## 1. Load and explore the data

The wine dataset contains 178 samples, 13 chemical measurements per sample, and
a target with 3 classes (the cultivar). We load it into a pandas DataFrame so we
can look at it as a feature table.

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data            # 13 chemical features
y = wine.target          # cultivar: 0, 1, 2

df = X.copy()
df["cultivar"] = y
print("Shape (samples, features):", X.shape)
print("\nFeatures:", list(X.columns))
print("\nClass counts:\n", y.value_counts().sort_index())
df.head()

### A quick look at two features

A scatter of two chemical measurements, coloured by cultivar, already shows that
the classes are partly separable. This is the kind of structure a model will
exploit.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for c in sorted(y.unique()):
    m = (y == c)
    ax.scatter(X.loc[m, "flavanoids"], X.loc[m, "color_intensity"], label=f"cultivar {c}", alpha=0.7)
ax.set_xlabel("flavanoids")
ax.set_ylabel("color_intensity")
ax.set_title("Two chemical features, coloured by cultivar")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Split into training and test sets

We hold back 30 % of the data for testing. `stratify=y` keeps the class
proportions the same in both sets. The model never sees the test set during
training, so its accuracy there is a first, rough estimate of how it would do on
new samples.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
print("train:", X_train.shape[0], "samples |  test:", X_test.shape[0], "samples")

## 3. A single decision tree (baseline)

A decision tree splits the data with yes/no questions on the features. It is our
simple, readable baseline. Note how a deep tree fits the training data almost
perfectly but does less well on the test set: a first taste of overfitting.

In [ ]:
tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

print("train accuracy:", round(accuracy_score(y_train, tree.predict(X_train)), 3))
print("test  accuracy:", round(accuracy_score(y_test,  tree.predict(X_test)), 3))

## 4. Random forest

A random forest averages many decision trees, each trained on a random subset of
the data and features. It is usually more accurate and more robust than one tree,
and it reports which features it relied on most.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\n", classification_report(y_test, y_pred, target_names=[f"cultivar {c}" for c in [0,1,2]]))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(cmap="Blues")
plt.title("Random forest - confusion matrix (test set)")
plt.show()

### Which chemical measurements matter most?

Feature importances rank the inputs by how much the forest used them. In a
chemistry context this is a useful sanity check: do the important features make
chemical sense?

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(6, 5))
importances.plot.barh(ax=ax)
ax.set_title("Random forest feature importances")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

## 5. SVM and why scaling matters

An SVM is sensitive to the scale of the features: a measurement with large
numeric values (like proline) would otherwise dominate. We compare an SVM on raw
features with the same SVM after standardising the features with a pipeline.
Watch the accuracy change.

In [ ]:
svm_raw = SVC(kernel="rbf", random_state=RANDOM_STATE)
svm_raw.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, svm_raw.predict(X_test))

svm_scaled = make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=RANDOM_STATE))
svm_scaled.fit(X_train, y_train)
acc_scaled = accuracy_score(y_test, svm_scaled.predict(X_test))

print("SVM test accuracy, raw features   :", round(acc_raw, 3))
print("SVM test accuracy, scaled features:", round(acc_scaled, 3))

## 6. Unsupervised exploration with k-means

So far we used the labels (supervised learning). k-means ignores the labels and
tries to find groups on its own. We ask for 3 clusters and then check, out of
curiosity, how well the discovered clusters line up with the true cultivars.
Scaling is required because k-means uses distances.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
clusters = km.fit_predict(X_scaled)

print("Cluster vs. true cultivar (rows = cluster, cols = cultivar):")
print(pd.crosstab(clusters, y))

coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(coords[:, 0], coords[:, 1], c=clusters, cmap="viridis", alpha=0.7)
ax[0].set_title("k-means clusters (PCA view)")
ax[1].scatter(coords[:, 0], coords[:, 1], c=y, cmap="viridis", alpha=0.7)
ax[1].set_title("true cultivars (PCA view)")
for a in ax:
    a.set_xlabel("PC1"); a.set_ylabel("PC2")
plt.tight_layout()
plt.show()

## 7. Reflection

Answer briefly in the cell below (these mirror the kind of thinking assessed
later in the course):

1. Which model gave the best test accuracy here? Was the extra complexity worth
   it over the decision-tree baseline?
2. The decision tree reached very high *training* accuracy. Why is that not, by
   itself, good news?
3. Scaling changed the SVM result but would not change the random forest. Why?
4. k-means was never given the labels, yet its clusters partly matched the
   cultivars. What does that suggest about the data, and why should you still be
   cautious about reading too much into it?

*Write your answers here.*
